# Index-Linked Hybrid Validation — Pucci (2012b)

**Reference:** M. Pucci, *Pricing Index-Linked Hybrids under a Local Volatility Market Model*, Banca IMI, 2012. SSRN 2056277.

Visual validation: price sensitivity to correlation, martingale checks, rho monotonicity.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import math
import numpy as np
import matplotlib.pyplot as plt
from pricebook.index_linked_hybrid import index_linked_hybrid_price
from pricebook.cash_settlement import cash_annuity

T = 5.0; N = 10
YFS = [0.5] * N; TAUS = [0.5*(i+1) for i in range(N)]
F0 = 0.04; U0 = 0.04; DF = math.exp(-0.03 * T)
SIGMA_F = 0.30; SIGMA_U = 0.25

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12})
print("Setup complete.")

In [ ]:
# 1. Price vs Correlation — ATM hybrid
rhos = np.linspace(-0.95, 0.95, 20)
prices_payer = []
prices_receiver = []
for rho in rhos:
    r = index_linked_hybrid_price(F0, U0, DF, YFS, TAUS, SIGMA_F, SIGMA_U,
                                   rho=rho, T=T, theta=1, n_paths=30_000, seed=42)
    prices_payer.append(r.price)
    r2 = index_linked_hybrid_price(F0, U0, DF, YFS, TAUS, SIGMA_F, SIGMA_U,
                                    rho=rho, T=T, theta=-1, n_paths=30_000, seed=42)
    prices_receiver.append(r2.price)

fig, ax = plt.subplots()
ax.plot(rhos, np.array(prices_payer)*10000, "b-o", ms=4, label="Payer ($\\theta=+1$)")
ax.plot(rhos, np.array(prices_receiver)*10000, "r-s", ms=4, label="Receiver ($\\theta=-1$)")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Correlation $\\rho$")
ax.set_ylabel("Price (bp of notional)")
ax.set_title("Index-Linked Hybrid Price vs Correlation (ATM: $F_0=U_0$)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Cash annuity Â(S) as function of swap rate
rates = np.linspace(0.005, 0.10, 100)
annuities = [cash_annuity(s, YFS, TAUS) for s in rates]

fig, ax = plt.subplots()
ax.plot(rates * 100, annuities, "b-", lw=2)
ax.set_xlabel("Swap Rate S (%)")
ax.set_ylabel("Cash Annuity $\\hat{A}(S)$")
ax.set_title("Cash Annuity: Decreasing Convex Function of S (Eq 3)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()